# 

# Generalization Datasets

- Classificaiton
  - tox21
  - toxcast
  - muv
  - pcba
- Regression
  - hopv - homo, lumo
  - zinc15 - logp
  - freesolv - hydration free energy
- Rxn
  - open reaction database
    - presto dataset( not compare with presto, because we assume OOD comparison)
- M2T
  - hanbum's dataset
- T2M
  - hanbum's dataset

# Check dataset availability

In [29]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets


def mol2graph(mol):
    """
    Converts SMILES string to graph Data object
    :input: SMILES string (str)
    :return: graph object
    """
    # atoms
    atom_features_list = []
    for atom in mol.GetAtoms():
        atom_features_list.append(atom_to_feature_vector(atom))
    x = np.array(atom_features_list, dtype = np.int64)

    # bonds
    num_bond_features = 3  # bond type, bond stereo, is_conjugated
    if len(mol.GetBonds()) > 0: # mol has bonds
        edges_list = []
        edge_features_list = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_feature = bond_to_feature_vector(bond)

            # add edges in both directions
            edges_list.append((i, j))
            edge_features_list.append(edge_feature)
            edges_list.append((j, i))
            edge_features_list.append(edge_feature)

        # data.edge_index: Graph connectivity in COO format with shape [2, num_edges]
        edge_index = np.array(edges_list, dtype = np.int64).T

        # data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
        edge_attr = np.array(edge_features_list, dtype = np.int64)

    else:   # mol has no bonds
        edge_index = np.empty((2, 0), dtype = np.int64)
        edge_attr = np.empty((0, num_bond_features), dtype = np.int64)

    graph = dict()
    graph['edge_index'] = edge_index
    graph['edge_feat'] = edge_attr
    graph['node_feat'] = x
    graph['num_nodes'] = len(x)

    return graph 

from rdkit import Chem
import selfies as sf
from download_dataset import wrap_label

system_prompt = "You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation."

def prepare_data_instance(
        mol,
        label,
        task,
        instruction_templates,
        system_prompt,
        mol_token="<mol>",
        num_query_tokens=32,
):

    label = wrap_label(label, task=task)
    input_prompt = np.random.choice(instruction_templates).item()
    assert "<INPUT>" in input_prompt, f"llm_prompt should contain <INPUT>"
    graph_sequence = "<GRAPH>" + mol_token * num_query_tokens + "</GRAPH>"

    if isinstance(mol, list):
        smiles = Chem.MolToSmiles(mol[0])
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        additional_smiles = Chem.MolToSmiles(mol[1])
        additional_selfies = sf.encoder(additional_smiles)
        additional_input_mol_string = "<SELFIES> " + additional_selfies + " </SELFIES>"
        additional_input_mol_string_graph = additional_input_mol_string + graph_sequence

        input_mol_string = input_mol_string + "|>>|" + additional_input_mol_string
        input_mol_string_graph = input_mol_string_graph + "|>>|" + additional_input_mol_string_graph
        
        graph = mol2graph(mol[0])
        additional_graph = mol2graph(mol[1])
    else:
        smiles = Chem.MolToSmiles(mol)
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        graph = mol2graph(mol)
        additional_graph = mol2graph(mol)
    
    input_prompt = input_prompt.replace("<INPUT>", input_mol_string_graph)

    formatted_prompt_text = "<s>[INST] " + system_prompt + " \n\n" + input_prompt + " [INST]"
    formatted_target_text = label + " </s>"

    data = {
        "task": task,
        "x": graph['node_feat'],
        "edge_index": graph['edge_index'],
        "edge_attr": graph['edge_feat'],
        "additional_x": additional_graph['node_feat'],
        "additional_edge_index": additional_graph['edge_index'],
        "additional_edge_attr": additional_graph['edge_feat'],
        "input_mol_string": input_mol_string,
        "prompt_text": formatted_prompt_text,
        "target_text": formatted_target_text,
    }
    return data


def get_data_list(
        list_mol, list_label, task, instruction_templates, system_prompt
):
    list_data = []
    iter_bar = tqdm(range(len(list_mol)))

    for i in iter_bar:
        data = prepare_data_instance(
        mol=list_mol[i],
        label=list_label[i], 
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )  
        list_data.append(data)
    return list_data


In [30]:
from torch_geometric.datasets import ZINC

task_name = "hopv"
base_path = f"/text-mol/dataset/{task_name}"
# Load the dataset
dataset = ZINC(root=base_path, split='train')  # Options: 'train', 'val', 'test'

# Access a single graph
data = dataset[0]
print(data)

Extracting data/ZINC/molecules.zip
Processing...
Processing test dataset: 100%|██████████| 5000/5000 [00:00<00:00, 11279.43it/s]


Data(x=[33, 1], edge_index=[2, 72], edge_attr=[72], y=[1])


Done!


In [49]:

from datasets import load_dataset

dataset = load_dataset("OpenMol/RCR_RP_57K_SMILES-MMChat")
task_name = "presto-reagent_prediction"
instruction_templates = instructions_smol.reagent_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(dataset['test'])))
for i in iter_bar:
    smiles = dataset['test'][i]['molecules']['smiles']
    label = dataset['test'][i]['ground_truth']
    if isinstance(smiles, list) and len(smiles) > 1:
        mol = [Chem.MolFromSmiles(s) for s in smiles]
    else:
        mol = Chem.MolFromSmiles(smiles[0])
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_dataset = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_dataset), len(omitted_idx))
dataset = datasets.Dataset.from_list(list_dataset)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(dataset[0])

100%|██████████| 6377/6377 [00:10<00:00, 615.67it/s]


6377 1


Saving the dataset (1/1 shards): 100%|██████████| 6377/6377 [00:00<00:00, 134622.55 examples/s]

{'task': 'presto-reagent_prediction', 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0]

In [48]:
from datasets import load_dataset

dataset = load_dataset("OpenMol/MolInst_FS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-forward_reaction_prediction"
instruction_templates = instructions_smol.forward_reaction_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(dataset['test'])))
for i in iter_bar:
    smiles = dataset['test'][i]['molecules']['smiles']
    label = dataset['test'][i]['ground_truth']
    if isinstance(smiles, list) and len(smiles) > 1:
        mol = [Chem.MolFromSmiles(s) for s in smiles]
    else:
        mol = Chem.MolFromSmiles(smiles[0])
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_dataset = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_dataset), len(omitted_idx))
dataset = datasets.Dataset.from_list(list_dataset)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(dataset[0])

  0%|          | 0/1004 [00:00<?, ?it/s][07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
 12%|█▏        | 120/1004 [00:00<00:00, 1195.73it/s][07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbors
[07:53:28] WARNING: not removing hydrogen atom without neighbor

1004 0


Saving the dataset (1/1 shards): 100%|██████████| 1004/1004 [00:00<00:00, 72685.05 examples/s]

{'task': 'presto-forward_reaction_prediction', 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0]], 'edge_index': [[0, 1, 1, 2, 1, 3, 3, 4, 3, 5], [1, 0, 2, 1, 3, 1, 4, 3, 5, 3]], 'edge_attr': [[1, 0, 1], [1, 0, 1], [0, 0, 0], [0, 0, 0], [0, 0, 1], [0, 0, 1], [1, 0, 1], [1, 0, 1], [0, 0, 0], [0, 0, 0]], 'additional_x': [[7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 2, 5, 1, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0]], 'additional_edge_index': [[0, 1, 1, 2, 1, 3, 3, 4, 4, 5, 5, 6, 5, 7, 5, 8], [1, 0, 2, 1, 3, 1, 4, 3, 5, 4, 6, 5, 7, 5, 8, 5]], 'additional_edge_attr': [[1, 0, 1], [1, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0

In [46]:
from datasets import load_dataset

dataset = load_dataset("OpenMol/MolInst_RS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-retrosynthesis"
instruction_templates = instructions_smol.retrosynthesis

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(dataset['test'])))
for i in iter_bar:
    smiles = dataset['test'][i]['molecules']['smiles']
    label = dataset['test'][i]['ground_truth']
    if isinstance(smiles, list) and len(smiles) > 1:
        mol = [Chem.MolFromSmiles(s) for s in smiles]
    else:
        mol = Chem.MolFromSmiles(smiles[0])
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_dataset = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
print(len(list_dataset), len(omitted_idx))
dataset = datasets.Dataset.from_list(list_dataset)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}")

print(dataset[0])

100%|██████████| 1000/1000 [00:01<00:00, 688.10it/s]


1000 0


Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 58041.40 examples/s]

{'task': 'presto-retrosynthesis', 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 4, 5, 2, 0, 2, 0, 0], [7, 0, 2, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [6, 0, 2, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [6, 0, 3, 5, 2, 0, 1, 0, 0], [6, 0, 2, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1]], 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 4, 8, 8, 9, 9, 10, 10, 11, 10, 12, 12, 13, 13, 14, 14, 15, 1

In [12]:
import deepchem as dc
import numpy as np

task_name = "qm7"
base_path = f"/text-mol/dataset/{task_name}"

tasks, dataset, transformers = dc.molnet.load_qm7(
                featurizer="Raw",
                splitter="scaffold",
                save_dir=base_path,
                data_dir=base_path,
                reload=True,
            )

# class arugments
list_mol = [dataset[i].X for i in range(3)]
list_mol = np.concatenate(list_mol)
list_y = [dataset[i].y for i in range(3)]
list_y = np.concatenate(list_y)

[12:00:16] Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] ERROR: Could not sanitize molecule ending on line 2301
[12:00:16] ERROR: Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] Explicit valence for atom # 2 N, 4, is greater than permitted
[12:00:16] ERROR: Could not sanitize molecule ending on line 2443
[12:00:16] ERROR: Explicit valence for atom # 2 N, 4, is greater than permitted
[12:00:16] Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] ERROR: Could not sanitize molecule ending on line 3284
[12:00:16] ERROR: Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] ERROR: Could not sanitize molecule ending on line 6828
[12:00:16] ERROR: Explicit valence for atom # 3 N, 4, is greater than permitted
[12:00:16] Explicit valence for atom # 2 N, 4, is greater than permitted
[12:00:16] ERROR: Could not sanitize molecule endin

In [2]:
import deepchem as dc
import instructions_smol
import numpy as np

task_name = "hopv"
base_path = f"/text-mol/dataset/{task_name}"

tasks, hopv_dataset, transformers = dc.molnet.load_hopv(
                featurizer="Raw",
                splitter="scaffold",
                save_dir=base_path,
                data_dir=base_path,
                reload=True,
            )

# class arugments
list_mol = [hopv_dataset[i].X for i in range(3)]
list_mol = np.concatenate(list_mol)
list_y = [hopv_dataset[i].y for i in range(3)]
list_y = np.concatenate(list_y)

In [4]:
import datasets

subtask_id = 0
task_name = "hopv_homo"
list_label = list_y[:, subtask_id]
instruction_templates = instructions_smol.qm9_homo

list_homo = get_data_list(
    list_mol=list_mol,
    list_label=list_label, 
    task=task_name, 
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
hopv_homo_dataset = datasets.Dataset.from_list(list_homo)

100%|██████████| 350/350 [00:00<00:00, 460.27it/s]


In [5]:
subtask_id = 1
task_name = "hopv_lumo"
list_label = list_y[:, subtask_id]
instruction_templates = instructions_smol.qm9_lumo

list_lumo = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
    system_prompt=system_prompt
)
hopv_lumo_dataset = datasets.Dataset.from_list(list_lumo)

100%|██████████| 350/350 [00:00<00:00, 509.68it/s]


In [7]:
llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_hopv_lumo")
hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_hopv_lumo")
hopv_lumo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_hopv_lumo")

hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_hopv_homo")
hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_hopv_homo")
hopv_homo_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_hopv_homo")

Saving the dataset (1/1 shards): 100%|██████████| 350/350 [00:00<00:00, 32836.16 examples/s]
